# Preprocesamiento: Valores Numéricos Faltantes (*Missing Values*)

## Pasos previos

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.model_selection import train_test_splithousing = pd.read_csv("./data/housing.csv") train_set, test_set = train_test_split(housing, test_size=0.2,    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]),    random_state=42    )

El **preprocesamiento** de datos es una de las tareas más importantes en Machine Learning. Si los datos no están bien preparados, los algoritmos de *Machine Learning* no funcionarán correctamente. Primero separaremos los **predictores** de la **variable objetivo** (las **etiquetas**), ya que no necesariamente aplicaremos las mismas **transformaciones** a ambos.

In [ ]:
X_train = train_set.drop("median_house_value", axis=1) # Eliminar la columna de la variable dependiente
y_train = train_set["median_house_value"].copy() # Guardar la variable dependiente (etiquetas)
X_train.head().T

## Identificación de valores faltantes

Como vimos al principio, la columna `total_bedrooms` tiene valores faltantes. Normalmente hablaremos de valores faltantes, **valores nulos** o **NA** (not available) como sinónimos, aunque hay que tener cuidado con la forma en que se recopilaron esos valores, ya que si existen dos tipos de valores (nulo y cadena vacía, por ejemplo) podría haber información implícita.

> **Nota terminológica:** En ciencia de datos, varios términos se usan de forma más o menos intercambiable:
> - **Valor faltante**: El concepto general: un valor que debería existir pero no existe.
> - **NA (Not Available)**: Una etiqueta común para los valores faltantes, usada en R y en pandas (`pd.NA`).
> - **NaN (Not a Number)**: Originalmente del estándar IEEE 754 de punto flotante, usado en NumPy/pandas para flotantes faltantes.
> - **Nulo**: Común en bases de datos (SQL NULL); en pandas, `None` para columnas de tipo objeto.
> - **None**: Valor de Python.
> 
> En pandas, `isna()` e `isnull()` son alias: ambos detectan valores NA y NaN. La distinción importante es entre datos *estructuralmente* faltantes (no registrados) y datos *semánticamente* faltantes (por ejemplo, "no aplica"), ya que pueden requerir estrategias de manejo distintas.

In [ ]:
X_train.isna().sum() # el método isnull() es un alias de isna()

In [ ]:
null_rows_idx = X_train.isnull().any(axis=1) # índices de las filas con valores nulos
X_train.loc[null_rows_idx].head()

## Los mecanismos de los datos faltantes

En ciencia de datos no solo preguntamos *qué* falta, sino *por qué* falta. Este "por qué" se denomina mecanismo de ausencia. Comprenderlo evita sacar conclusiones estadísticamente "alucinadas".

### 1. Faltante Completamente al Azar (MCAR, *Missing Completely at Random*)

En MCAR, la ausencia es un proceso de "ruido blanco". La probabilidad de que un dato falte es constante, independientemente de cualquier valor en el conjunto de datos, ya sea observado o no observado.

* **La lógica:** Los datos observados son un reflejo perfecto, aunque más pequeño, del total.
* **Formalismo:** $P(\text{Ausencia} \mid \text{Todo}) = P(\text{Ausencia})$
* **Sobre la imputación:** Aunque se *pueden* eliminar estas filas sin sesgar la media, la **imputación** suele ser preferida. ¿Por qué? Porque eliminar filas desperdicia los datos válidos en otras columnas, reduciendo la potencia estadística. Imputar (por ejemplo, con la mediana) ayuda a mantener el conjunto de datos "completo", aunque puede reducir artificialmente la varianza.
* **Ejemplo:** Un sensor pierde energía durante 5 minutos por un fallo en la red eléctrica local. La temperatura que debería haber registrado no influyó en el fallo eléctrico.

### 2. Faltante al Azar (MAR, *Missing at Random*)

MAR es el término más confuso en estadística porque suena como "aleatorio", pero en realidad significa **sistemático, aunque explicable**.

* **La confusión:** Cuando decimos "al azar", no queremos decir que la ausencia sea como lanzar una moneda. Queremos decir que **una vez que se tienen en cuenta los datos que SÍ observamos, la incertidumbre restante es aleatoria**.
* **La lógica:** La "causa" del hueco está registrada en otra columna ($X$). Si se observa el conjunto de datos completo, hay un patrón claro. Pero si se analiza un subgrupo específico (por ejemplo, solo las mujeres), el patrón desaparece y la ausencia parece un accidente puro.
* **Formalismo:** $P(\text{Ausencia} \mid X, Y) = P(\text{Ausencia} \mid X)$
  > Esto significa: "Si conozco $X$, $Y$ no me aporta nada nuevo sobre por qué faltan los datos."
* **Ejemplo:** En una encuesta de salud, las mujeres ($X$) tienen estadísticamente menos probabilidad de reportar su peso ($Y$) que los hombres. Sin embargo, dentro del grupo de mujeres, la decisión de omitir la pregunta no está determinada por su *peso real*, sino por una tendencia demográfica. Si sabemos que la persona encuestada es mujer, hemos capturado la "causa" de los datos faltantes.
* **Solución:** Usar las variables observadas ($X$) para "rellenar" los huecos en ($Y$).

### 3. Faltante No al Azar (MNAR, *Missing Not at Random*)

En MNAR, la probabilidad de ausencia depende de **información no observada**. Esta información puede ser el propio valor faltante, o una variable que simplemente no se midió.

* **¿Dónde está la información?** La distinción entre MAR y MNAR radica en la **ubicación de la causa**:
    * **MAR:** La causa está en el conjunto de datos (en otra columna).
    * **MNAR:** La causa está **fuera** del conjunto de datos (ya sea en el propio valor faltante o en una variable no observada que no se registró).
* **Ejemplo:** Las personas con una deuda muy alta tienen menos probabilidad de reportar su nivel de deuda en una solicitud de préstamo. O un termómetro que simplemente no puede registrar temperaturas superiores a 100°C y devuelve un valor nulo en su lugar.
* **Implicación:** Este es el tipo más "peligroso". La imputación estándar falla porque los datos observados no contienen la información necesaria para explicar el hueco. Se requieren modelos especializados (como la corrección de Heckman) o análisis de sensibilidad específicos del dominio.

### Tabla resumen

| Tipo | Fuente de información de la ausencia | Estrategia |
| :--- | :--- | :--- |
| **MCAR** | Ninguna (Ruido) | Imputar o eliminar |
| **MAR** | Otras variables observadas | Imputación múltiple (usar $X$ para predecir $Y$) |
| **MNAR** | El propio valor faltante | Modelado especializado / Modelos de selección |

## Eliminación de filas con valores nulos (***Listwise deletion***)

Podemos simplemente eliminar las instancias incompletas, aunque esto es problemático porque estamos eliminando información. Especialmente si hay muchos predictores (ya que al resolver el problema de ciertos valores nulos estamos perdiendo la información de las otras columnas).

In [ ]:
X_train_ld_tb = X_train.dropna(subset=["total_bedrooms"]) 
X_train_ld_tb.loc[null_rows_idx].head() # verificar que las filas con valores nulos han sido eliminadas

También podríamos eliminar directamente cualquier fila que tenga un valor nulo en alguna columna:

In [ ]:
X_train_ld_any = X_train.dropna(axis=0) # eliminar filas con valores nulos
X_train_ld_any.loc[null_rows_idx].head() # verificar que las filas con valores nulos han sido eliminadas

## Eliminación de la columna completa

Eliminar la columna completa es una opción si no es una variable importante, pero en este caso parece ser importante dado que, aunque esa *característica* no es la que más directamente correlaciona con la variable objetivo, es una de las dos utilizadas para calcular `bedrooms_ratio`, que es la segunda con mayor correlación.

In [ ]:
X_train_drop_tb = X_train.drop(columns="total_bedrooms")
X_train_drop_tb.loc[null_rows_idx].head()

Las filas siguen estando en este caso porque los índices nulos se buscaron antes. Si buscamos nulos ahora en `housing_option2`, no los encontraremos.

In [ ]:
X_train_drop_tb.isnull().any(axis=None) # verificar que no hay valores nulos en el dataset

También podríamos eliminar directamente todas las columnas con valores nulos:

In [ ]:
X_train.dropna(axis=1).isnull().any(axis=None)

np.False_

## Imputación de algún valor (la mediana en este caso)

La **imputación** de un determinado valor (como cero, la media o la mediana) a los campos faltantes es una opción si creemos que los valores faltantes no responden a ninguna causa específica y no sesgan la distribución de la variable.

En nuestro conjunto de datos, `total_bedrooms` tiene solo 207 valores faltantes (~1%) de un total de 20.640 registros (una fracción pequeña). Sin información adicional sobre *por qué* faltan estos valores, la imputación por la mediana es una elección razonable, ya que es robusta ante posibles valores atípicos.

La imputación por la media (***mean***) es más sensible a los **valores atípicos**, ya que un valor extremo puede afectar considerablemente a la media (consulta [valores atípicos y valores acotados](e2e020_eda.ipynb#outliers-and-capped-values) para técnicas de manejo de valores atípicos). La mediana (***median***) es más robusta ante valores extremos. La moda (***mode***) es el valor que más se repite y es útil para variables categóricas, pero no tanto para variables continuas.

[<img src="./img/mean_outliers.jpg" width="300">](https://www.kaggle.com/code/nareshbhat/outlier-the-silent-killer)

In [ ]:
median = X_train["total_bedrooms"].median()
housing_option3 = X_train["total_bedrooms"].fillna(median)
housing_option3.loc[null_rows_idx].head()

Ahora todas estas filas tienen en `total_bedrooms` el valor de la mediana de `total_bedrooms`.

La clase `SimpleImputer` de scikit-learn nos permite hacer esto de forma más sencilla. Creamos una instancia de `SimpleImputer` indicando que queremos imputar los valores nulos con la mediana, y luego usamos el método `fit()` para calcular la mediana de cada columna y el método `transform()` para aplicar la imputación a todas las columnas.

Veamos cómo se aplicaría este método a todos los campos numéricos del dataframe (recuerda que `ocean_proximity` es categórico —valores de texto— y no podemos calcular la mediana de texto).

In [ ]:
from sklearn.impute import SimpleImputerimputer = SimpleImputer(strategy="median")

In [ ]:
housing_num = X_train.select_dtypes(include=[np.number]) # seleccionar columnas numéricas

In [ ]:
imputer.fit(housing_num) # calcular la mediana de cada columna numérica
imputer.statistics_ # mediana de cada columna numérica

Podemos verificar que los valores son los mismos que los calculados por el método `median()` del dataframe.

In [ ]:
housing_num.median().values

array([-118.51   ,   34.26   ,   29.     , 2119.     ,  433.     ,
       1164.     ,  408.     ,    3.54155])

In [ ]:
housing_num_array_tr = imputer.transform(housing_num) # reemplazar valores nulos con la mediana
housing_num_array_tr

`transform()` devuelve un array de NumPy, pero podríamos convertirlo de nuevo a un dataframe de pandas.

In [ ]:
housing_tr = pd.DataFrame(housing_num_array_tr, columns=housing_num.columns, index=housing_num.index)housing_tr.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
1606,-122.08,37.88,26.0,2947.0,433.0,825.0,626.0,2.9330
10915,-117.87,33.73,45.0,2264.0,433.0,1970.0,499.0,3.4193
19150,-122.70,38.35,14.0,2313.0,433.0,954.0,397.0,3.7813
4186,-118.23,34.13,48.0,1308.0,433.0,835.0,294.0,4.2891
16885,-122.40,37.58,26.0,3281.0,433.0,1145.0,480.0,6.3580


También podríamos usar directamente el método `fit_transform()` de `SimpleImputer` para calcular el valor a imputar (con `fit()`) y aplicarlo (con `transform()`) en un solo paso. Y podríamos también usar el método `.set_output(transform="pandas")` del imputer para que el resultado sea un dataframe de pandas.

Por tanto, todo el proceso detallado anteriormente se puede resumir en una sola línea de código:

In [ ]:
housing_tr = SimpleImputer(strategy="median").set_output(transform="pandas").fit_transform(housing_num)housing_tr.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
1606,-122.08,37.88,26.0,2947.0,433.0,825.0,626.0,2.9330
10915,-117.87,33.73,45.0,2264.0,433.0,1970.0,499.0,3.4193
19150,-122.70,38.35,14.0,2313.0,433.0,954.0,397.0,3.7813
4186,-118.23,34.13,48.0,1308.0,433.0,835.0,294.0,4.2891
16885,-122.40,37.58,26.0,3281.0,433.0,1145.0,480.0,6.3580


## Modelos predictivos para imputar valores

Existen métodos más avanzados, como el uso de **modelos de predicción** (tratando la columna con valores nulos como la variable objetivo y el resto de las columnas como *características*). Por ejemplo, el algoritmo **K-Nearest Neighbors (KNN)** podría usarse para predecir los valores nulos de `total_bedrooms` basándose en los registros etiquetados. Scikit-Learn dispone de una clase `KNNImputer` que hace esto.